In [ ]:
from collections import defaultdict
import glob
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [ ]:
with open("../../data/accession_translator.json", "r") as s:
    accession_translator = json.load(s)

In [ ]:
thresholds = defaultdict(float)
for log in glob.glob("../../intermediates/geneml_scores/*.log"):
    with open(log, "r") as s:
        for line in s:
            if "Dynamic threshold calculated" in line:
                threshold = float(re.search(r'Dynamic threshold calculated:\s+([\d.]+)', line).group(1))
                break
    accession = log.split("/")[-1].replace(".log", "")
    thresholds[accession] = threshold

In [ ]:
paths = glob.glob("../../intermediates/geneml_scores/*_scores.txt")
dfs = []
for path in paths:
    accession = path.split("/")[-1].replace("_scores.txt", "")
    df = pd.read_csv(path, header=None, names=['score','length'], delimiter='\t')
    df["species"] = accession_translator[accession]
    dfs.append(df)
all_scores = pd.concat(dfs)
all_scores["species"] = pd.Categorical(all_scores["species"], accession_translator.values())

In [ ]:
g = sns.displot(data=all_scores, y="length", x="score", col="species", col_wrap=3, cmap='Reds', log_scale=(False, True), bins=(25, 20), common_norm=False, binrange=((0, 1), None))
g.set(xlim=(0, 1))

# Map species names to dynamic thresholds via accession
species_thresholds = {
    accession_translator[acc]: thr
    for acc, thr in thresholds.items()
    if acc in accession_translator
}

for i, ax in enumerate(g.axes.flat):
    species = ax.get_title().split(" = ", 1)[-1]
    threshold = species_thresholds.get(species)
    if threshold is None:
        continue
    if i == 0:
        ax.axvline(x=threshold, color='black', linestyle='--', linewidth=1, label='Dynamic score cutoff')
    else:
        ax.axvline(x=threshold, color='black', linestyle='--', linewidth=1)

g.fig.legend(loc='lower left', bbox_to_anchor=(0.05, -0.02), fontsize=12)
g.fig.tight_layout()
plt.savefig("../../figures/Figure_S1.svg")